#   Preprocessing

In [1]:
pip install joblib scipy seaborn numpy pandas matplotlib scikit-learn lightgbm xgboost plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import joblib

# Data Manipulation
import numpy as np
import pandas as pd
from scipy.sparse import csc_matrix, eye, diags
from scipy.sparse.linalg import spsolve

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder,
    normalize,
    OneHotEncoder
)

# Model Selection & Metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from lightgbm import LGBMClassifier
import xgboost as xgb

# Decomposition & Math
from sklearn.decomposition import PCA, KernelPCA
from numpy.polynomial.polynomial import polyfit, polyval
from scipy.signal import savgol_filter
from scipy import stats


# Extras
from sklearn.inspection import permutation_importance

In [3]:
def load_raman_mapping(file_path):
    # Read file as text first to filter invalid rows
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    valid_lines = []
    for line in lines:
        parts = line.strip().split()
        # Keep only rows with exactly 4 numeric values
        if len(parts) == 4:
            try:
                [float(x) for x in parts]  # Check all values are numeric
                valid_lines.append(line)
            except ValueError:
                continue  # Skip rows with non-numeric values
    
    # Check if any valid lines remain BEFORE parsing
    if len(valid_lines) == 0:
        return None, None
    
    # Create DataFrame from valid lines only
    from io import StringIO
    df = pd.read_csv(StringIO(''.join(valid_lines)), sep=r'\s+', header=None)
    
    df.columns = ['X', 'Y', 'Wave', 'Intensity']

    spectra = df.pivot_table(
        index=['X', 'Y'], 
        columns='Wave', 
        values='Intensity'
    )

    X_raw = spectra.values
    wavenumbers = spectra.columns.values

    return X_raw, wavenumbers

In [4]:
def zscore_mask_only(X_numpy):
    X_safe = X_numpy.astype(np.float64).copy()
    X_safe = np.nan_to_num(X_safe)

    z_scores = np.abs(stats.zscore(X_safe, axis=1))
    max_z = np.max(z_scores, axis=1)

    threshold = np.percentile(max_z, 95)
    keep_mask = max_z < threshold

    kept_pct = keep_mask.mean() * 100
    print(f"Threshold={threshold:.2f}")

    return keep_mask

In [5]:
def parsing_data(dir_path):
    data_list = []

    for dir_1 in os.listdir(dir_path):
        for dir_2 in os.listdir(f'{dir_path}/{dir_1}'):
            for dir_3 in os.listdir(f'{dir_path}/{dir_1}/{dir_2}'):
                if not dir_3.endswith('.txt'): 
                    continue
                    
                print(f'{dir_3}')
                center, cells = None, None
                for i in dir_3.split('_'):
                    if 'center' in i:
                        center = i
                    if 'cortex' in i or 'striatum' in i or 'cerebellum' in i:
                        cells = i

                X_raw, wavenumbers = load_raman_mapping(f'{dir_path}/{dir_1}/{dir_2}/{dir_3}')
                
                #  Skip files with no valid data
                if X_raw is None:
                    print(f"  ⚠ Skipped (no valid rows)")
                    continue

                metadata_df = pd.DataFrame({
                    'center': [center] * X_raw.shape[0],
                    'cells': [cells] * X_raw.shape[0], 
                    'cat': [dir_1] * X_raw.shape[0]
                })

                file_data = pd.concat([pd.DataFrame(X_raw), metadata_df], axis=1)
                data_list.append(file_data)

    if len(data_list) == 0:
        raise ValueError("No valid data files found")
    
    return pd.concat(data_list, ignore_index=True)

In [6]:
data = parsing_data('data')

striatum_left_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place3_3.txt
cortex_left_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place2_1.txt
cortex_left_control_2Agroup_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place3_1.txt
striatum_right_control_2Agroup_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place2_4.txt
striatum_left_control_2Agroup_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place2_3.txt
striatum_right_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place1_4.txt
cortex_left_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place3_1.txt
striatum_left_control_2Agroup_633nm_center2900_obj100_power100_1s_5acc_map35x15_step2_place3_3.txt
striatum_left_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place1_3.txt
striatum_right_control_2Agroup_633nm_center1500_obj100_power100_1s_5acc_map35x15_step2_place3_4.txt
striatum_righ

In [49]:
df = data

In [50]:
label_encoder_cat = LabelEncoder()
df['cat'] = label_encoder_cat.fit_transform(data['cat'])

In [51]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = ohe.fit_transform(df[['center', 'cells']])

In [52]:
df = pd.concat([df.drop(['center', 'cells'], axis=1), pd.DataFrame(cat_encoded, columns=['center', 'cell1', 'cell2'])], axis=1)

In [53]:
X, y = df.drop('cat', axis=1), df['cat']

In [54]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.drop(['center', 'cell1', 'cell2'], axis=1))
X_scaled = X_scaled.astype(np.float32)

In [55]:
# X_normalized = normalize(X_scaled, norm='l2')

In [56]:
pca = PCA(n_components=0.95, svd_solver='auto', random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Start


In [63]:
# Проверка на NaN перед расчётом
print(f"NaN in X_pca: {np.isnan(X_pca).sum()}")
X_pca = np.nan_to_num(X_pca, nan=0.0)  # Заменить NaN на 0

keep_mask = zscore_mask_only(X_pca)

NaN in X_pca: 0
Threshold=nan


In [65]:
X = pd.concat([
    pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])]),
    df[['center', 'cell1', 'cell2']].reset_index(drop=True)
], axis=1)

In [67]:
X = X[keep_mask].reset_index(drop=True)
y = y[keep_mask].reset_index(drop=True)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

ValueError: Item wrong length 124425 instead of 0.

In [61]:
X

,0,center,cell1,cell2


In [ ]:
# Проверка
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.drop(['center', 'cell1', 'cell2'], axis=1),  # Только PCA признаки
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]}')
print(f'Test:  {X_test.shape[0]}')

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    verbose=0
)

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.025,
    max_depth=15,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    class_weight='balanced',
    verbose=-1,
    n_jobs=-1
)

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)

In [ ]:
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)

result = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)


print(f"RF Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test, rf_pred, target_names=['сontrol','endo','exo']))

RF Accuracy: 0.743
              precision    recall  f1-score   support

     сontrol       0.71      0.75      0.73      7902
        endo       0.79      0.75      0.77      7080
         exo       0.74      0.73      0.73      8659

    accuracy                           0.74     23641
   macro avg       0.75      0.74      0.74     23641
weighted avg       0.74      0.74      0.74     23641



In [ ]:
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)

result = permutation_imresult = permutation_importance(lgb_model, X_test, y_test, n_repeats=10, random_state=42)
result = permutation_importance(lgb_model, X_test, y_test, n_repeats=10, random_state=42)


print(f"LGB Accuracy: {accuracy_score(y_test, lgb_pred):.3f}")
print(classification_report(y_test, lgb_pred, target_names=['сontrol','endo','exo']))

LGB Accuracy: 0.736
              precision    recall  f1-score   support

     сontrol       0.71      0.74      0.72      7902
        endo       0.78      0.74      0.76      7080
         exo       0.72      0.73      0.73      8659

    accuracy                           0.74     23641
   macro avg       0.74      0.74      0.74     23641
weighted avg       0.74      0.74      0.74     23641



In [ ]:
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)

result = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42)

print(f"XGB Accuracy: {accuracy_score(y_test, xgb_pred):.3f}")
print(classification_report(y_test, xgb_pred, target_names=['сontrol','endo','exo']))

XGB Accuracy: 0.926
              precision    recall  f1-score   support

     сontrol       0.92      0.92      0.92      7900
        endo       0.94      0.92      0.93      7219
         exo       0.92      0.93      0.92      8522

    accuracy                           0.93     23641
   macro avg       0.93      0.93      0.93     23641
weighted avg       0.93      0.93      0.93     23641



RF Accuracy: 0.848
              precision    recall  f1-score   support

     сontrol       0.84      0.86      0.85      7900
        endo       0.86      0.83      0.85      7219
         exo       0.84      0.85      0.85      8522

    accuracy                           0.85     23641
   macro avg       0.85      0.85      0.85     23641
weighted avg       0.85      0.85      0.85     23641

LGB Accuracy: 0.916
              precision    recall  f1-score   support

     сontrol       0.91      0.92      0.91      7900
        endo       0.92      0.92      0.92      7219
         exo       0.91      0.91      0.91      8522

    accuracy                           0.92     23641
   macro avg       0.92      0.92      0.92     23641
weighted avg       0.92      0.92      0.92     23641

XGB Accuracy: 0.926
              precision    recall  f1-score   support

     сontrol       0.92      0.92      0.92      7900
        endo       0.94      0.92      0.93      7219
         exo       0.92      0.93      0.92      8522

    accuracy                           0.93     23641
   macro avg       0.93      0.93      0.93     23641
weighted avg       0.93      0.93      0.93     23641